In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F

from torch import optim
from torch import nn
from torch.utils.data import random_split, DataLoader
from tqdm import tqdm

import torchvision

import torchvision.datasets as datasets
import torchvision.transforms as transforms
import torchvision.models as models

import torchmetrics

In [2]:
# Obtain sampled datasets of benign and malignant tumor images
# Run 'sampling.ipynb' first to generate the data in 'samples/'

# --- your paths ---
path = "C:\\Users\\alvin\\OneDrive\\Documents\\CS171\\project\\breast-cancer-classifier-cs171-s02" # change variable as needed

samples = path + "/samples/"

benign_samples = path + "/samples/benign/"
malignant_samples = path + "/samples/malignant/"

In [16]:
# Define batch size
batch_size = 150

# Compress images and convert to Tensor to create transform function
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

# Retrieve the sampled dataset from the directory. ImageFolder assigns classes to images (benign = 0; malignant = 1)
dataset = datasets.ImageFolder(root=samples, transform=transform)

# Debug: check if class labels are correct
print(dataset.classes)
print(dataset.class_to_idx)

# Define split sizes (70% training, 15% validation, 15% test)
total = len(dataset)
training_size = int(0.7 * total)
val_size = int(0.15 * total)
test_size = total - training_size - val_size # avoid rounding issues

# Split the dataset into training set, validation set, and test set.
train_set, val_set, test_set = torch.utils.data.random_split(dataset, [training_size, val_size, test_size])

# Load each set using DataLoader. Shuffle only the training set, not the validation and test sets
train_loader = DataLoader(dataset=train_set, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(dataset=val_set, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(dataset=test_set, batch_size=batch_size, shuffle=False)

['benign', 'malignant']
{'benign': 0, 'malignant': 1}


In [17]:
# Load the pre-trained ResNeXt-50 model
model = models.resnext50_32x4d(weights=models.ResNeXt50_32X4D_Weights.DEFAULT, progress=True)

# Keep everything but the classification head, which will classify images based on two class labels
in_features = model.fc.in_features  # 2,048 output features from ResNeXt-50's CNN model
model.fc = nn.Sequential(
    nn.Linear(in_features, 256),    # Fully-connected layer that compresses 2,048 -> 256 features
    nn.ReLU(),                      # An activation function to add non-linearity
    nn.Dropout(0.25),               # Drop 25% of neurons randomly to reduce overfitting of data
    nn.Linear(256, 2)               # Classify benign or malignant
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Function to train model with validation
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs=10):
    for epoch in range(epochs):
        # --- Training ---
        model.train()
        train_loss, train_acc = 0, 0

        # Train model on training set
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device) # images and labels are either sent to CUDA or CPU
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            # Calculate the loss function to check the prediction of breast cancer images on training set
            train_loss += loss.item()
            # Accumulate all the correct predictions across all batches
            train_acc += (outputs.argmax(1) == labels).sum().item()
        
        # --- Validation ---
        model.eval()
        val_loss, val_acc = 0, 0

        # To save memory, avoid tracking gradient descent on the validation set
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)

                # Calculate the loss function to check the prediction of breast cancer images on validation set
                val_loss += loss.item()
                # Accumulate all the correct predictions across all batches
                val_acc += (outputs.argmax(1) == labels).sum().item()

        # --- Statistics ---
        print(f"Epoch {epoch+1}/{epochs}")
        print(f"  Train Loss: {train_loss/len(train_loader):.4f} | Train Acc: {train_acc/len(train_loader.dataset):.4f}")
        print(f"  Val Loss:   {val_loss/len(val_loader):.4f}   | Val Acc: {val_acc/len(val_loader.dataset):.4f}")
        
criterion = nn.CrossEntropyLoss()                           # Use Cross Entropy Loss function
optimizer = torch.optim.Adam(model.fc.parameters(), lr=0.1) # Use Adam optimizer with a learning rate of 0.1

train_model(model, train_loader, val_loader, criterion, optimizer, epochs=10)

In [ ]:
# Evaluate the model on a test set
# --- Testing ---
model.eval()
test_acc = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        test_acc += (outputs.argmax(1) == labels).sum().item()

print(f"Test Accuracy: {test_acc/len(test_loader.dataset):.4f}")